# 3D inductive field

`InductiveField3D` solves the time-harmonic magnetoquasistatic field of coils and conductors
in 3D, on the edges of a Cartesian grid (`EdgeGrid`), for geometries without a symmetry,
such as several drivers with their shields. The discretisation is in
[3D field discretisation](/wellbalanced/development/field-3d/). This tutorial checks it on
a long solenoid in a conducting box, whose inductance per length is known in closed form:
$L' = \mu_0 n^2 A_s (1 - A_s / A_\mathrm{box})$.

In [ ]:
import numpy as np
import plasma_plots  # noqa: F401, the .plasma accessor
import xarray as xr
from scipy.constants import mu_0
from simgeometry import Cylinder

from wellbalanced.spatial import Coil3D, EdgeGrid, InductiveField3D

half, radius, turns, depth = 0.1, 0.05, 20.0, 0.02  # m, m, 1/m, m
box_area = (2 * half) ** 2
exact = mu_0 * turns**2 * np.pi * radius**2 * (1 - np.pi * radius**2 / box_area)


def solenoid(cells):
    """A solenoid along z through a box, periodic along z (an infinite solenoid)."""
    grid = EdgeGrid(
        (-half, -half, 0.0),
        (half, half, depth),
        (cells, cells, 2),
        ("dirichlet", "dirichlet", "periodic"),
    )
    coil = Coil3D(Cylinder((0, 0, -1), (0, 0, 1), radius), (0, 0, 1), turns)
    return InductiveField3D(grid, frequency=1e3, coils=[coil])


model = solenoid(20)
solution = model.solve(1.0)  # 1 A
solution.inductance / depth, exact

The coil is a current sheet on the surface of its region, `J = curl T`; on the grid the
circle is staircased, so the error shrinks with the cells:

In [ ]:
cells = [10, 20, 40]
error = xr.DataArray(
    [abs(solenoid(n).solve(1.0).inductance / depth / exact - 1) for n in cells],
    coords={"cells": ("cells", cells, {"long_name": "cells across the box"})},
    name="error",
    attrs={"long_name": "relative error of L'"},
)
error.plasma.plot.convergence()

Conductors of a given conductivity (`Conductor3D`), good conductors as surface impedances
(`ImpedanceConductor3D`) and a background conductivity such as a cold plasma's complex
$\sigma$ (`conductivity=`) add their losses to the solution: `solution.loss`,
`solution.conductor_losses` and `solution.resistance`.